EGM722 Programming for GIS and Remote Sensing Assessment 2

Title: Mapping geothermal energy opportunity in relation to deep geothermal play attributes and aquifer geology in Northern Ireland using Python.

Aim: This project assesses indicative geothermal suitability using British Geological Survey (BGS) deep geothermal play attributes and aquifer geology.

1 Import Python libraries

In [ ]:
import sys

!{sys.executable} -m pip install geopandas folium

1 Import Python libraries

In [ ]:
# File and folder handling
from pathlib import Path
import os
import zipfile
import urllib.request

# Data analysis
import pandas as pd

# Spatial data handling
import geopandas as gpd

# Mapping and plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Interactive web mapping
import folium

# Spatial geometry operations
from shapely.geometry import box

# Suppress warnings for cleaner notebook output
import warnings
warnings.filterwarnings("ignore")

# Display all dataframe columns in notebook outputs
pd.set_option("display.max_columns", 100)

2 Set file paths

This section defines the folder structure used throughout the project.

Separate folders are used for:
- raw downloaded datasets,
- processed GIS outputs,
- and exported figures/results.

Using a consistent folder structure improves reproducibility and organisation of spatial analysis workflows. The folders are automatically created if they do not already exist.

The project uses pathlib.Path because it provides a platform-independent and reliable method for handling file paths in Python.

In [ ]:
from pathlib import Path

# Project folders
PROJECT_DIR = Path.cwd().parent

DATA_RAW = PROJECT_DIR / "data_raw"
DATA_PROCESSED = PROJECT_DIR / "data_processed"
OUTPUTS = PROJECT_DIR / "outputs"

# Create folders if needed
for folder in [DATA_RAW, DATA_PROCESSED, OUTPUTS]:
    folder.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data folder:", DATA_RAW)
print("Processed data folder:", DATA_PROCESSED)
print("Outputs folder:", OUTPUTS)

3 Set Coordinate Reference Systems (CRS)

In [ ]:
# British National Grid is used by the BGS deep geothermal shapefile
BNG_CRS = "EPSG:27700"

# Irish National Grid is useful for NI environmental datasets
ING_CRS = "EPSG:29903"

# WGS84 is required for Folium web maps
WEB_CRS = "EPSG:4326"

# Use BNG as the analysis CRS because the BGS geothermal layer is supplied in BNG
TARGET_CRS = BNG_CRS

4 Define functions

In [ ]:
def download_file(url, output_path):
    """
    Download a file from a URL if it does not already exist.

    Parameters
    ----------
    url : str
        URL of the file to download.

    output_path : pathlib.Path
        Location where the file will be saved.

    Returns
    -------
    pathlib.Path
        Path to the downloaded file.
    """

    # Skip download if file already exists locally
    if output_path.exists():
        print(f"File already exists: {output_path.name}")
        return output_path

    print(f"Downloading: {url}")

    urllib.request.urlretrieve(url, output_path)

    print(f"Saved to: {output_path}")

    return output_path

def load_vector(path, target_crs=TARGET_CRS, layer=None):
    """
    Load a vector GIS dataset and reproject it to a target CRS.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to the GIS file, for example SHP, GeoJSON or GeoPackage.
    target_crs : str
        Coordinate reference system to reproject the dataset into.
    layer : str, optional
        Layer name to read when using a GeoPackage with multiple layers.

    Returns
    -------
    geopandas.GeoDataFrame
        Loaded and reprojected spatial dataset.
    """
    # Read the spatial file using GeoPandas.
    gdf = gpd.read_file(path, layer=layer)

    # Stop the workflow if the dataset has no CRS.
    if gdf.crs is None:
        raise ValueError(f"{path} has no CRS defined.")

    # Reproject to the chosen CRS so all datasets align spatially.
    return gdf.to_crs(target_crs)


def clean_geodataframe(gdf):
    """
    Clean a GeoDataFrame by removing missing geometries and repairing invalid ones.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Input spatial dataset.

    Returns
    -------
    geopandas.GeoDataFrame
        Cleaned spatial dataset.
    """
    # Work on a copy so the original data is not modified accidentally.
    gdf = gdf.copy()

    # Remove rows without valid geometry.
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]

    # Repair invalid polygon geometries using a zero-width buffer.
    gdf["geometry"] = gdf.geometry.buffer(0)

    return gdf

def classify_geothermal_play(row):
    """
    Assign an indicative geothermal suitability score using BGS geothermal play attributes.

    The scoring is based on general geothermal reasoning:
    sedimentary basins, hydrothermal systems, and fault/fracture-controlled settings
    are considered more favourable than poorly defined or low-permeability settings.

    Parameters
    ----------
    row : pandas.Series
        One row from the geothermal GeoDataFrame.

    Returns
    -------
    int
        Suitability score from 1 to 5, where 5 is highest.
    """
    text = " ".join([
        str(row.get("GTPlayType", "")),
        str(row.get("GeoHabitat", "")),
        str(row.get("GeoControls", "")),
        str(row.get("BGSPlayDesc", "")),
        str(row.get("Status", ""))
    ]).lower()

    score = 1

    # Sedimentary basins are often favourable for hydrothermal geothermal resources
    if "sedimentary" in text or "basin" in text:
        score += 2

    # Hydrothermal systems are directly relevant to geothermal production
    if "hydrothermal" in text:
        score += 1

    # Faults and fractures can increase permeability
    if "fault" in text or "fracture" in text:
        score += 1

    # Sandstone and limestone can be favourable aquifer lithologies
    if "sandstone" in text or "limestone" in text:
        score += 1

    # Granites may be relevant for deep geothermal heat but often need fracture permeability
    if "granite" in text:
        score += 1

    return min(score, 5)


def classify_aquifer_lithology(description):
    """
    Score aquifer geology based on simplified lithological suitability.

    More porous aquifer materials are given higher scores because they may
    store and transmit groundwater more effectively.

    Parameters
    ----------
    description : str
        Aquifer geology, lithology or description field.

    Returns
    -------
    int
        Aquifer suitability score from 0 to 5.
    """
    if pd.isna(description):
        return 0

    text = str(description).lower()

    if "sandstone" in text:
        return 5
    elif "limestone" in text or "chalk" in text:
        return 4
    elif "basalt" in text or "volcanic" in text:
        return 3
    elif "granite" in text:
        return 2
    elif "schist" in text or "gneiss" in text or "metamorphic" in text:
        return 1
    else:
        return 2


def classify_final_score(score):
    """
    Convert combined suitability score into a class.

    Parameters
    ----------
    score : float
        Combined geothermal suitability score.

    Returns
    -------
    str
        Suitability class.
    """
    if score >= 4:
        return "High"
    elif score >= 3:
        return "Moderate"
    elif score >= 2:
        return "Low"
    else:
        return "Very low"

5 Download and extract datasets from URLs

5.1 Download and extract the Aquifer data

In [ ]:
PROJECT_DIR = Path.cwd().parent
DATA_RAW = PROJECT_DIR / "data_raw"
DATA_RAW.mkdir(exist_ok=True)

AQUIFER_URL = "https://gsni-data.bgs.ac.uk/geonetwork/srv/api/records/eeee2244-953d-4b81-8124-9e1ce602bf2e/attachments/NorthernIrelandsGroundwaterEnvironment.zip"

AQUIFER_ZIP = DATA_RAW / "aquifers.zip"
AQUIFER_EXTRACTED = DATA_RAW / "aquifers_extracted"

# Download the ZIP file
urllib.request.urlretrieve(AQUIFER_URL, AQUIFER_ZIP)

# Extract the ZIP file
AQUIFER_EXTRACTED.mkdir(exist_ok=True)

with zipfile.ZipFile(AQUIFER_ZIP, "r") as zip_ref:
    zip_ref.extractall(AQUIFER_EXTRACTED)

# List extracted files
for file in AQUIFER_EXTRACTED.rglob("*"):
    print(file)

5.2 Locate the Aquifer Geopackage from the zip file

In [ ]:
# Find GeoPackage files inside extracted folder
gpkg_files = list(AQUIFER_EXTRACTED.rglob("*.gpkg"))

print("GeoPackage files found:")
for file in gpkg_files:
    print(file)

AQUIFER_FILE = gpkg_files[0]
print("Using:", AQUIFER_FILE)

5.3 Download and extract the BGS Deep Geothermal Areas data

In [ ]:
PROJECT_DIR = Path.cwd().parent
DATA_RAW = PROJECT_DIR / "data_raw"
DATA_RAW.mkdir(exist_ok=True)

GEOTHERMAL_URL = "https://webservices.bgs.ac.uk/accessions/download/185538?fileName=Deep_Geothermal_Areas_v2.zip"

GEOTHERMAL_ZIP = DATA_RAW / "Deep_Geothermal_Areas_v2.zip"
GEOTHERMAL_EXTRACTED = DATA_RAW / "deep_geothermal_areas_v2"

# Download ZIP file
urllib.request.urlretrieve(GEOTHERMAL_URL, GEOTHERMAL_ZIP)

# Extract ZIP file
GEOTHERMAL_EXTRACTED.mkdir(exist_ok=True)

with zipfile.ZipFile(GEOTHERMAL_ZIP, "r") as zip_ref:
    zip_ref.extractall(GEOTHERMAL_EXTRACTED)

# List extracted files
for file in GEOTHERMAL_EXTRACTED.rglob("*"):
    print(file)

5.4 Locate the BGS Deep Geothermal Areas Shapefile from the zip file

In [ ]:
shp_files = list(GEOTHERMAL_EXTRACTED.rglob("*.shp"))

print("Shapefiles found:")
for file in shp_files:
    print(file)

GEOTHERMAL_FILE = shp_files[0]
print("Using:", GEOTHERMAL_FILE)

5.5 Load Aquifer and BGS Deep Geothermal Areas data into GeoPandas

In [ ]:
aquifer = load_vector(AQUIFER_FILE)
geothermal = load_vector(GEOTHERMAL_FILE)

print("Aquifer records loaded:", len(aquifer))
print("Geothermal areas loaded:", len(geothermal))

display(aquifer.head())
display(geothermal.head())

6 Exploratory data analysis

6.1 Inspect the structure of the data

6.2 Check CRS

6.3 Summary statistics

7 Data cleaning

8 Feature selection

9 Check for missing data

10 Save the cleaned data

11 Reproject data to the same CRS

12 Clip datasets to Northern Ireland

13 Spatial analysis

14 Output map

15 Interactive map

16 Export outputs

17 Conclusions and limitations